# Module 2 Guided Lab: Building a Multi-Tool ReAct Agent
### Updated for LangChain v1.3.14

**Estimated time:** ~2 hours  
**What you'll build:** A LangChain agent (using `create_agent()`) that can search the web and run Python code — then harden it with error handling and prompt engineering.

---

## How this notebook is structured

There are four steps. Each step has:
- **Pre-written code** — run it as-is to understand what's happening
- **Extension zones** — marked `# STUDENT EXTENSION POINT` — where you add or modify code
- **A self-check** at the end to verify your output before moving on

> **Rule:** Only modify code inside extension zones. The scaffolding is designed so each step teaches one concept in isolation — changing code outside these zones makes it harder to diagnose what went wrong.

---

## Agent Instruction Log

Throughout this lab you will record observations in an **Agent Instruction Log**. This is a running document (a text file, Google Doc, or a Markdown cell in this notebook) where you answer specific questions after each step. Look for 📝 **Log Entry** callouts — these tell you exactly what to record and under what heading.

The log is a portfolio artifact. It is submitted alongside your `.ipynb` file.

---

## Version Notes (LangChain v1.3.14)

This notebook uses **LangChain v1.3.14** and the modern `create_agent()` API. Key differences from the legacy approach:
- `create_agent()` replaces the old `create_react_agent()` + `AgentExecutor` pattern
- It returns a compiled `StateGraph` (LangGraph under the hood) — you call `.invoke()` or `.stream()` on it
- Tools are plain Python functions with docstrings — no `Tool()` wrapper needed
- The system prompt is a simple string — no `PromptTemplate` with special placeholders
- The agent automatically handles the tool-calling loop until no more tool calls remain
- Use `debug=True` for verbose tracing (replaces `verbose=True` on `AgentExecutor`)

---
# Step 1 — Set Up Your Environment and Run a Basic Agent
**Time:** ~30 minutes

By the end of Step 1 you will have a working agent that can search the web.

### What is a tool-calling agent?
LangChain's `create_agent()` builds a graph that runs a loop:

1. **Model call** — the LLM receives the conversation and decides whether to call a tool
2. **Tool execution** — if the model requested tool calls, they are executed
3. **Observation** — tool results are added back to the messages

This loop repeats until the model produces a response with **no tool calls** (i.e., a final answer). This is conceptually the same as the ReAct pattern (Yao et al., 2022) but implemented via native tool-calling rather than text-parsed `Thought/Action/Observation` formatting.

### Platform requirements
- Google account (for Colab — free tier is sufficient)
- **OpenAI API key**, OR a locally hosted **Ollama** model (free alternative; see Section 1B below)

In [ ]:
# Install required packages. Run this cell once and wait for it to finish.
# You should see a list of installed packages with no red error text.
!pip install -q \
    langchain>=1.3.14 \
    langchain-openai \
    langchain-community \
    duckduckgo-search>=7.0.0

### Section 1A — OpenAI API Key Setup

Paste your OpenAI API key in the cell below. The key starts with `sk-`.

> **Security note:** Never share a notebook with your key visible. In Colab, use the Secrets panel (🔑 icon in the left sidebar) and reference it as `userdata.get('OPENAI_API_KEY')` for a safer alternative.

### Section 1B — Ollama Alternative (free, no API key needed)

If you do not have an OpenAI key, skip to the **Ollama Setup** cell below and follow those instructions instead. Skip the OpenAI cell.

In [ ]:
import os

# STUDENT EXTENSION POINT — Step 1: Add your API key here
os.environ["OPENAI_API_KEY"] = "sk-YOUR-KEY-HERE"

# Verify the key is set (prints True if it starts with 'sk-')
print("API key set:", os.environ.get("OPENAI_API_KEY", "").startswith("sk-"))

In [ ]:
# SECTION 1B — Ollama Alternative
# Only run this cell if you are NOT using OpenAI.
# Skip entirely if you completed Section 1A.
#
# Steps:
#   1. Install Ollama from https://ollama.com
#   2. Run `ollama pull llama3` in your terminal
#   3. Uncomment the lines below and run this cell

# !pip install -q langchain-ollama
# from langchain_ollama import ChatOllama
# llm = ChatOllama(model="llama3", temperature=0)
# print("Ollama LLM ready:", llm.invoke("Say hello in one word.").content)

In [ ]:
# Initialize the LLM (OpenAI path).
# If using Ollama, comment out these two lines — the llm variable was set above.
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Quick connectivity test — should print a short reply
print(llm.invoke("Reply with exactly three words: model is ready.").content)

In [ ]:
# Build a single-tool web-search agent using create_agent().
# Pre-written — read every comment before running.

from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchRun

# --- Tool definition ---
# With create_agent(), tools can be plain functions with docstrings.
# The docstring IS the tool description — it's the only signal the LLM uses
# to decide when to call this tool. It must be precise.
search = DuckDuckGoSearchRun()

def duckduckgo_search(query: str) -> str:
    """Use this tool to search the web for current information, recent events,
    or factual data that may not be in the model's training data.
    Input: a plain English search query of 5-10 words.
    Output: a text snippet from web search results.
    Do not use this tool for mathematical calculations."""
    return search.run(query)

tools = [duckduckgo_search]

# --- System prompt ---
# With create_agent(), the system prompt is a plain string.
# No special placeholders needed — the agent handles tool routing automatically.
SYSTEM_PROMPT = (
    "You are a helpful research assistant. Use the available tools to answer the user's "
    "question accurately. Always search for current information before making factual claims. "
    "When you have enough information to answer, provide a complete and well-structured response."
)

# --- Agent assembly ---
# create_agent() returns a compiled StateGraph.
# debug=True prints the full execution trace (similar to verbose=True in the old API).
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    debug=True
)

print("Agent ready.")
print("Tools:", [t.__name__ if callable(t) else t.name for t in tools])

In [ ]:
# Run the agent on the test prompt.
# With debug=True you will see the full execution trace in the output.

result = agent.invoke({
    "messages": [{"role": "user", "content": "What were the three most significant AI research papers published in the last 30 days?"}]
})

# The result contains the full message history.
# The final message is the agent's answer.
print("\n--- FINAL ANSWER ---")
print(result["messages"][-1].content)

In [ ]:
# Helper function to display the full agent trace in a readable format.
# This shows the reasoning loop: model calls, tool calls, tool results.

def print_agent_trace(result):
    """Print the full message trace from an agent invocation."""
    for i, msg in enumerate(result["messages"]):
        role = msg.__class__.__name__
        if role == "HumanMessage":
            print(f"\n[{i}] USER: {msg.content}")
        elif role == "AIMessage":
            if msg.tool_calls:
                print(f"\n[{i}] ASSISTANT (tool calls):")
                for tc in msg.tool_calls:
                    print(f"    → {tc['name']}({tc['args']})")
            else:
                print(f"\n[{i}] ASSISTANT (final answer):")
                print(f"    {msg.content[:500]}")
        elif role == "ToolMessage":
            content_preview = msg.content[:200] + "..." if len(msg.content) > 200 else msg.content
            print(f"\n[{i}] TOOL ({msg.name}): {content_preview}")
        elif role == "SystemMessage":
            print(f"\n[{i}] SYSTEM: {msg.content[:100]}...")

print("--- FULL TRACE ---")
print_agent_trace(result)

### ✅ Step 1 Self-Check

Look at the trace output above. You should see this pattern:

| Message Type | What it means |
|---|---|
| `USER` | The original question |
| `ASSISTANT (tool calls)` | The model decided to call `duckduckgo_search` |
| `TOOL (duckduckgo_search)` | The raw text returned by the search tool |
| `ASSISTANT (final answer)` | The model's synthesized response |

**If the tool was never called:** The system prompt text `'Always search for current information before making factual claims'` must be present — verify it above.

**If DuckDuckGo returns an error:** Wait 30 seconds and re-run (free tier has rate limits).

---

### 📝 Log Entry — Step 1 (record in your Agent Instruction Log)

**Heading:** `Step 1 — Single-Tool Agent`

Answer these four questions:
1. **Tool call:** What search query did the model generate? (Copy from the trace)
2. **Tool invoked:** Which tool was called?
3. **Observation summary:** What did the search return? (1–2 sentences)
4. **Loop count:** Did the agent call the tool once or multiple times before producing its final answer?

---
# Step 2 — Multi-Tool Extension: Adding a Python REPL
**Time:** ~40 minutes

A search-only agent cannot compute. In this step you add a **Python REPL tool** — a tool that lets the agent write and execute Python code inside Colab and observe the result. This makes the agent capable of combining retrieval (search) with computation (REPL).

### Why tool descriptions matter
The LLM has no inherent knowledge of when to use search vs. REPL. It infers the right tool entirely from the **docstring/description text**. A vague description leads to wrong tool selection. You will see this directly in Action 8 below.

In [ ]:
# STUDENT EXTENSION POINT — Step 2
# Task: Uncomment the REPL tool function below, then add it to the tools list.
# The tool code is pre-written — you only need to (1) uncomment it and (2) update tools=[...].

import subprocess
import sys

# --- Uncomment this function ---
# def python_repl(code: str) -> str:
#     """Use this tool for numerical calculations, data manipulation, and Python code execution.
#     Input: valid Python code as a string.
#     Output: the printed output or result of the code.
#     Do not use this tool to look up current facts or search the web."""
#     try:
#         result = subprocess.run(
#             [sys.executable, "-c", code],
#             capture_output=True, text=True, timeout=30
#         )
#         output = result.stdout
#         if result.stderr:
#             output += "\nSTDERR: " + result.stderr
#         return output.strip() if output.strip() else "(no output)"
#     except subprocess.TimeoutExpired:
#         return "Error: Code execution timed out (30s limit)"
#     except Exception as e:
#         return f"Error: {str(e)}"

# Update the tools list to include both tools.
# Change this line:
tools = [duckduckgo_search]
# To this (after uncommenting the python_repl function above):
# tools = [duckduckgo_search, python_repl]

# Rebuild the agent with the updated tools list.
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    debug=True
)
print("Agent rebuilt with tools:", [t.__name__ for t in tools])

In [ ]:
# Run all three test prompts in sequence.
# Do not modify the prompts — they are designed to test specific tool selection behaviors.

prompts = {
    "A": "What is the current population of Tokyo, and what is 3.7% of that number?",
    "B": "Calculate the compound interest on a $10,000 principal at 5.25% annual rate over 8 years.",
    "C": "Find today's exchange rate between USD and EUR, then calculate how much EUR you would receive for $2,500."
}

results = {}
for label, prompt in prompts.items():
    print(f"\n{'='*60}")
    print(f"PROMPT {label}: {prompt}")
    print('='*60)
    result = agent.invoke({
        "messages": [{"role": "user", "content": prompt}]
    })
    results[label] = result
    print(f"\nFINAL ANSWER ({label}):")
    print(result["messages"][-1].content)
    print("\n--- TRACE ---")
    print_agent_trace(result)

### Action 8 — Deliberately Break Tool Selection

In the cell below, change the REPL tool docstring to something vague, then re-run Prompt A. Observe whether the agent's tool selection changes.

**Purpose:** This demonstrates that tool selection is controlled entirely by description text — not by task complexity or any inherent model knowledge.

In [ ]:
# STUDENT EXTENSION POINT — Step 2, Action 8
# Redefine python_repl with a deliberately vague docstring.
# Then rebuild the agent and re-run Prompt A.

# def python_repl_vague(code: str) -> str:
#     """Use for advanced tasks."""  # <-- intentionally vague
#     try:
#         result = subprocess.run(
#             [sys.executable, "-c", code],
#             capture_output=True, text=True, timeout=30
#         )
#         output = result.stdout
#         if result.stderr:
#             output += "\nSTDERR: " + result.stderr
#         return output.strip() if output.strip() else "(no output)"
#     except Exception as e:
#         return f"Error: {str(e)}"

# tools_vague = [duckduckgo_search, python_repl_vague]
# agent_vague = create_agent(
#     model=llm,
#     tools=tools_vague,
#     system_prompt=SYSTEM_PROMPT,
#     debug=True
# )

# result_vague = agent_vague.invoke({
#     "messages": [{"role": "user", "content": prompts["A"]}]
# })
# print("\nFINAL ANSWER (vague description):", result_vague["messages"][-1].content)
# print("\n--- TRACE ---")
# print_agent_trace(result_vague)

### ✅ Step 2 Self-Check

Before moving on, confirm:

- **Prompt A** invokes `duckduckgo_search` first (for population), then `python_repl` (for the 3.7% calculation)
- **Prompt B** invokes **only** `python_repl` — no web search. If the agent searches for a compound interest formula before computing, the REPL docstring is not explicit enough that math is in scope. Add `"Use this for any mathematical computation — no web lookup is needed"` to the docstring and re-run.
- **Prompt C** invokes search first (for the exchange rate), then REPL (for the multiplication)

---

### 📝 Log Entry — Step 2 (record in your Agent Instruction Log)

**Heading:** `Step 2 — Multi-Tool Agent`

Create a table with the following structure (one row per prompt):

| Prompt | Tools Invoked (in order) | Why the agent chose those tools | Did it match your prediction? |
|---|---|---|---|
| A | ... | ... | Yes / No |
| B | ... | ... | Yes / No |
| C | ... | ... | Yes / No |

Then add one more entry:
- **Action 8 result:** Did changing the REPL docstring to `'Use for advanced tasks.'` change which tool was selected for Prompt A? Why or why not?

---
# Step 3 — Error Handling: Deliberate Failure and Recovery
**Time:** ~30 minutes

Production agents must fail gracefully. In this step you will:
1. Trigger a real tool failure and observe the raw (unhandled) behavior
2. Implement an error handling wrapper that returns a structured error message instead of a Python traceback
3. Confirm the agent now responds usefully instead of crashing or hallucinating

### Why this matters
An unhandled exception in a tool produces a raw Python traceback in the tool result message. Most LLMs will either loop indefinitely trying to fix the code, or produce a hallucinated answer that ignores the failure. A structured error message gives the model actionable information to recover from.

In [ ]:
# Action 9 — Trigger a deliberate tool failure.
# Run this cell and observe what happens. Record the agent's behavior.
# Do NOT modify this prompt.

trigger_prompt = "Run the following code: import nonexistent_module; nonexistent_module.do_something()"

result_unhandled = agent.invoke({
    "messages": [{"role": "user", "content": trigger_prompt}]
})
print("\n--- AGENT RESPONSE (unhandled failure) ---")
print(result_unhandled["messages"][-1].content)
print("\n--- TRACE ---")
print_agent_trace(result_unhandled)

In [ ]:
# Action 10 — Error Handling Wrapper
# STUDENT EXTENSION POINT — Step 3
#
# Replace the raw REPL tool with a wrapped version that catches exceptions
# and returns a structured error message instead of a raw traceback.
#
# Instructions:
#   1. Read the wrapper function below — it adds structured error handling.
#   2. Uncomment the wrapped tool function.
#   3. Rebuild the agent using tools = [duckduckgo_search, python_repl_safe].
#      (NOT tools=[duckduckgo_search, python_repl] — the safe version must replace the original.)

import json

# Uncomment this function:
# def python_repl_safe(code: str) -> str:
#     """Use this tool for numerical calculations, data manipulation, and Python code execution.
#     Input: valid Python code as a string.
#     Output: the printed output, or a structured error message if the code fails.
#     Do not use this tool to look up current facts or search the web."""
#     try:
#         result = subprocess.run(
#             [sys.executable, "-c", code],
#             capture_output=True, text=True, timeout=30
#         )
#         if result.returncode != 0:
#             return json.dumps({
#                 "status": "error",
#                 "message": result.stderr.strip(),
#                 "recommendation": "Try a different approach or reformulate the query."
#             })
#         output = result.stdout.strip()
#         return output if output else "(no output)"
#     except subprocess.TimeoutExpired:
#         return json.dumps({
#             "status": "error",
#             "message": "Code execution timed out (30s limit)",
#             "recommendation": "Simplify the code or break it into smaller steps."
#         })
#     except Exception as e:
#         return json.dumps({
#             "status": "error",
#             "message": str(e),
#             "recommendation": "Try a different approach or reformulate the query."
#         })

# Rebuild with the safe version:
# tools = [duckduckgo_search, python_repl_safe]
# agent = create_agent(
#     model=llm,
#     tools=tools,
#     system_prompt=SYSTEM_PROMPT,
#     debug=True
# )
# print("Agent rebuilt with safe REPL:", [t.__name__ for t in tools])

In [ ]:
# Action 11 — Re-run the trigger prompt with the wrapped tool.
# The agent should now receive a structured error in its tool result
# and respond with either an alternative approach or an informative final answer.

result_handled = agent.invoke({
    "messages": [{"role": "user", "content": trigger_prompt}]
})
print("\n--- AGENT RESPONSE (with error wrapper) ---")
print(result_handled["messages"][-1].content)
print("\n--- TRACE ---")
print_agent_trace(result_handled)

### ✅ Step 3 Self-Check

After implementing the wrapper, the agent's final answer **must not** contain a raw Python traceback (lines starting with `Traceback (most recent call last):`). 

If a traceback still appears: the wrapper is not being applied. Check that `tools = [duckduckgo_search, python_repl_safe]` uses the safe version, not the original `python_repl`.

> **Common mistake:** Defining `python_repl_safe` but then passing `python_repl` (the original) to the `tools` list.

---

### 📝 Log Entry — Step 3 (record in your Agent Instruction Log)

**Heading:** `Step 3 — Error Handling`

Record the following:
- **(a) Before the wrapper:** What did the agent do when the tool failed? (Looped? Hallucinated? Produced a traceback in Final Answer?)
- **(b) After the wrapper:** How did the agent's behavior change? What appeared in the tool result?
- **(c) User utility assessment:** Was the graceful degradation response useful to a hypothetical end user? Why or why not? (2–3 sentences)

---
# Step 4 — Prompt Engineering: System Instructions and Few-Shot Demonstrations
**Time:** ~20 minutes

System instructions and few-shot examples are the primary controls for agent behavior. In this step you will:
1. Add an **output format constraint** to the system prompt and observe the change
2. Add a **few-shot demonstration** to the system prompt and assess whether the agent mimics it

### What is a few-shot demonstration?
A few-shot demonstration is a complete example of the interaction pattern you want the agent to follow, placed inside the system prompt. With `create_agent()`, you embed examples directly in the system prompt string showing the format you expect in the final answer.

In [ ]:
# Action 13 — Read the default system instruction.
# The default instruction is pre-written below.
# Do not modify it yet — read it carefully before Action 14.

DEFAULT_INSTRUCTION = (
    "You are a helpful research assistant. Use the available tools to answer the user's "
    "question accurately. Always search for current information before making factual claims."
)

print(DEFAULT_INSTRUCTION)

In [ ]:
# Action 14 — Add an output format constraint.
# STUDENT EXTENSION POINT — Step 4
#
# Replace YOUR_FORMAT_CONSTRAINT_HERE with a specific output structure requirement.
# Example (copy this or write your own):
#   "Always structure your final answer with: "
#   "(1) a direct answer in the first sentence, "
#   "(2) supporting evidence cited from your search results, "
#   "(3) a confidence assessment: Low, Medium, or High."

FORMAT_CONSTRAINT = "YOUR_FORMAT_CONSTRAINT_HERE"

UPDATED_INSTRUCTION = DEFAULT_INSTRUCTION + "\n\n" + FORMAT_CONSTRAINT

# Rebuild the agent with the updated system prompt
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=UPDATED_INSTRUCTION,
    debug=True
)

# Re-run Prompt A with the updated system instruction
result_formatted = agent.invoke({
    "messages": [{"role": "user", "content": prompts["A"]}]
})
print("\n--- FINAL ANSWER (with format constraint) ---")
print(result_formatted["messages"][-1].content)

In [ ]:
# Action 15 — Add a few-shot demonstration.
# STUDENT EXTENSION POINT — Step 4
#
# Write a complete example interaction showing the output format you expect.
# Use a DIFFERENT task than Prompts A, B, or C.
#
# With create_agent(), few-shot examples go directly in the system prompt as
# example Q&A pairs showing the desired response format.

# Section 4B — Few-Shot Template
# Fill in YOUR_FEW_SHOT_EXAMPLE with your own complete example.
# The example below is a placeholder — replace it with a task from your domain.
FEW_SHOT_EXAMPLE = """Here is an example of how to structure your response:

User question: "What is the GDP of France, and what is 2% of that number?"

Expected response format:
France's GDP is approximately $3.1 trillion (source: World Bank, 2024). 2% of that is $62 billion.

Evidence: World Bank data reports France's nominal GDP at ~$3.1 trillion USD for 2024.
Confidence: Medium (GDP figures vary by source and year)."""

FEW_SHOT_INSTRUCTION = UPDATED_INSTRUCTION + "\n\n" + FEW_SHOT_EXAMPLE

# Rebuild the agent with the few-shot system prompt
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=FEW_SHOT_INSTRUCTION,
    debug=True
)

# Run Prompt C with the few-shot prompt
result_fewshot = agent.invoke({
    "messages": [{"role": "user", "content": prompts["C"]}]
})
print("\n--- FINAL ANSWER (with few-shot) ---")
print(result_fewshot["messages"][-1].content)
print("\n--- TRACE ---")
print_agent_trace(result_fewshot)

### ✅ Step 4 Self-Check

**Format constraint:** Compare the final answer from Action 14 to the one from Step 2 (same Prompt A). Does the new answer follow the structure you specified?

**Few-shot demonstration:** Look at the final answer from Action 15. Does the agent's response structure resemble your example? If not, check that your few-shot example clearly shows the format you want and is included in the system prompt.

---

### 📝 Log Entry — Step 4 (record in your Agent Instruction Log)

**Heading:** `Step 4 — Prompt Engineering`

Record the following:
- **(a) Format constraint text:** Copy the exact text of your output format constraint
- **(b) Before/after comparison:** Quote the final answer from Step 2 (Prompt A, no constraint) and the final answer from Action 14 (Prompt A, with constraint). How did the structure change?
- **(c) Few-shot influence:** Did the agent's response in Action 15 mimic your demonstration's format? Cite specific evidence from the output.

---
# Lab Complete — Submission Instructions

Before submitting, confirm all four checks:

- [ ] All cells have been executed (no empty output cells)
- [ ] Your Agent Instruction Log contains entries for all four steps
- [ ] Step 3's final answer contains no raw Python tracebacks
- [ ] Step 4 shows a before/after comparison of final answer format

**Submit to GitHub:**
- File: `Module2_Unit3_Lab_[YourName].ipynb`
- All cells must be executed with visible output

---

## What's next?

Unit 4 builds directly on this lab. In Task 1 you will design your own complete prompt engineering suite — system instruction, tool descriptions, and few-shot demonstrations — for a domain of your choice. The agent architecture insights from Steps 2–4 are the inputs to that design task.

In [ ]:
# Optional: Use this cell as your Agent Instruction Log if you prefer to keep it in the notebook.
# Add your entries below — one section per step.

AGENT_INSTRUCTION_LOG = """
=== AGENT INSTRUCTION LOG ===

--- Step 1 — Single-Tool Agent ---
1. Tool call (search query used):
2. Tool invoked:
3. Observation summary:
4. Loop count:

--- Step 2 — Multi-Tool Agent ---
| Prompt | Tools Invoked | Why | Matched prediction? |
| A      |               |     |                     |
| B      |               |     |                     |
| C      |               |     |                     |

Action 8 result:

--- Step 3 — Error Handling ---
(a) Before wrapper:
(b) After wrapper:
(c) User utility assessment:

--- Step 4 — Prompt Engineering ---
(a) Format constraint text:
(b) Before/after Final Answer comparison:
(c) Few-shot influence:
"""

print(AGENT_INSTRUCTION_LOG)